# Local Search Experiments (Campaign-Time Calibration)

All experiments use campaign-time calibration with $\eta$ parameter for energy scaling.
The local search framework consists of:
- **6 initial tour heuristics** (R1–R6)
- **4 neighborhood operators** (add, replace, swap, 2-opt)
- **Metaheuristics:** Simulated Annealing, Iterated Local Search (ILS), Hill Climbing, Tilted Runs

In [ ]:
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from uav_routing.environment import Environment
from uav_routing.environment.calibration import calibrate, calibration_info
from uav_routing.environment.graph import Graph, directed_cycle, nearest_neighbor_tour
from uav_routing.environment.drone import Drone
from uav_routing.local_search.state import State
from uav_routing.local_search.optimization import Optimizer
from uav_routing.local_search.proposal import universal_proposal, random_flip_with_tabu
from uav_routing.local_search.accept import always_accept
from uav_routing.local_search.initial_tour import (
    tour_with_nearest_neighbors, tour_with_max_ratio,
    random_tour, random_then_nn_tour, longest_max_ratio
)
from uav_routing.solver.socp import Solver

# Dataset paths
datasets = {
    "C101 (50)":  "../datasets/data/50_c101.txt",
    "R101 (50)":  "../datasets/data/50_r101.txt",
    "RC101 (100)": "../datasets/data/rc101.txt",
    "R101 (100)": "../datasets/data/r101.txt",
    "C101 (100)": "../datasets/data/c101.txt",
}

SEED = 42
SORTIE_TIME = 3.0

def make_instance(path, slope='random', graph_seed=1, eta=1.0):
    """Create a campaign-calibrated instance."""
    graph = Graph(path=path, slope=slope, seed=graph_seed)
    uav = Drone(base=graph.graph['base'])
    calib = calibrate(
        graph=graph,
        tour_length=len(graph.nodes) // 2,
        drone=uav,
        drone_sortie_time=SORTIE_TIME,
        calibration_method="campaign",
    )
    return Environment(calib, uav, eta=eta), graph, uav


def build_initial_tour_R1(instance):
    """R1: Nearest-node tour of length n/2."""
    graph = instance.graph
    depot = instance.drone.base
    n_target = (len(graph.nodes) - 1) // 2
    tour = [depot]
    visited = {depot}
    current = depot
    for _ in range(n_target):
        candidates = [n for n in graph.nodes if n not in visited]
        if not candidates:
            break
        nearest = min(candidates, key=lambda n: graph.edges[(current, n)]['distance'])
        tour.append(nearest)
        visited.add(nearest)
        current = nearest
    return directed_cycle(tour, graph)


def build_initial_tour_R2(instance):
    """R2: Earliest-opening tour of length n/2."""
    graph = instance.graph
    depot = instance.drone.base
    n_target = (len(graph.nodes) - 1) // 2
    tour = [depot]
    visited = {depot}
    for _ in range(n_target):
        candidates = [n for n in graph.nodes if n not in visited]
        if not candidates:
            break
        earliest = min(candidates, key=lambda n: graph.nodes[n]['time_window'][0])
        tour.append(earliest)
        visited.add(earliest)
    return directed_cycle(tour, graph)


def build_initial_tour_R3(instance, rng=None):
    """R3: Random-then-nearest tour of length n/2."""
    if rng is None:
        rng = random.Random(SEED)
    graph = instance.graph
    depot = instance.drone.base
    n_target = (len(graph.nodes) - 1) // 2
    targets = [n for n in graph.nodes if n != depot]
    first = rng.choice(targets)
    tour = [depot, first]
    visited = {depot, first}
    current = first
    for _ in range(n_target - 1):
        candidates = [n for n in graph.nodes if n not in visited]
        if not candidates:
            break
        nearest = min(candidates, key=lambda n: graph.edges[(current, n)]['distance'])
        tour.append(nearest)
        visited.add(nearest)
        current = nearest
    return directed_cycle(tour, graph)


def build_initial_tour_R4(instance):
    """R4: Consumption-based tour (extends until infeasible)."""
    graph = instance.graph
    drone = instance.drone
    depot = drone.base
    tour = [depot]
    visited = {depot}
    current = depot
    while True:
        candidates = [n for n in graph.nodes if n not in visited]
        if not candidates:
            break
        best_node, best_cost = None, float('inf')
        for j in candidates:
            t_ratio = graph.edges[(current, j)]['distance'] / (drone.optimum_speed * instance.time_horizon)
            e_ratio = drone.energy_function(drone.optimum_speed, graph.edges[(current, j)]['distance']) / instance.max_energy
            cost = max(t_ratio, e_ratio)
            if cost < best_cost:
                best_cost = cost
                best_node = j
        # Check feasibility by solving SOCP with return to depot
        trial_tour = directed_cycle(tour + [best_node], graph)
        solver = Solver(trial_tour, instance)
        if solver.solution is None:
            break
        tour.append(best_node)
        visited.add(best_node)
        current = best_node
    return directed_cycle(tour, graph)


def build_initial_tour_R5(instance, rng=None, length=3):
    """R5: Random tour of 3 nodes."""
    if rng is None:
        rng = random.Random(SEED)
    graph = instance.graph
    depot = instance.drone.base
    targets = [n for n in graph.nodes if n != depot]
    chosen = rng.sample(targets, min(length, len(targets)))
    return directed_cycle([depot] + chosen, graph)


def build_initial_tour_R6(instance):
    """R6: Information-efficiency tour of length n/2."""
    graph = instance.graph
    drone = instance.drone
    depot = drone.base
    n_target = (len(graph.nodes) - 1) // 2
    tour = [depot]
    visited = {depot}
    current = depot
    for _ in range(n_target):
        candidates = [n for n in graph.nodes if n not in visited]
        if not candidates:
            break
        best_node, best_ratio = None, -float('inf')
        for j in candidates:
            info_j = graph.nodes[j]['info_at_lowest'] + graph.nodes[j]['info_slope'] * (
                graph.nodes[j]['time_window'][1] - graph.nodes[j]['time_window'][0])
            d_ij = graph.edges[(current, j)]['distance']
            d_j0 = graph.edges[(j, depot)]['distance']
            d_i0 = graph.edges[(current, depot)]['distance']
            delta_E = (drone.energy_function(drone.optimum_speed, d_ij)
                       + drone.energy_function(drone.optimum_speed, d_j0)
                       - drone.energy_function(drone.optimum_speed, d_i0))
            if delta_E > 0:
                ratio = info_j / delta_E
            else:
                ratio = float('inf') if info_j > 0 else 0
            if ratio > best_ratio:
                best_ratio = ratio
                best_node = j
        tour.append(best_node)
        visited.add(best_node)
        current = best_node
    return directed_cycle(tour, graph)


INITIAL_TOURS = {
    "R1 (nearest)": build_initial_tour_R1,
    "R2 (earliest)": build_initial_tour_R2,
    "R3 (rand+NN)": build_initial_tour_R3,
    "R4 (consumption)": build_initial_tour_R4,
    "R5 (random)": build_initial_tour_R5,
    "R6 (info-eff)": build_initial_tour_R6,
}

---
## 1. Initial Tour Comparison

Evaluate all 6 initial tour heuristics (R1–R6) on the same instance.
Each tour is evaluated by solving the SOCP subproblem for optimal speeds.

In [ ]:
test_path = datasets["R101 (50)"]
instance, graph, uav = make_instance(test_path)

init_rows = []
for name, builder in INITIAL_TOURS.items():
    t0 = time.time()
    tour_graph = builder(instance)
    build_time = time.time() - t0
    solver = Solver(tour_graph, instance)
    td = solver.get_tour_data()
    init_rows.append({
        "Heuristic": name,
        "Tour length": len(tour_graph.nodes) - 1,
        "Feasible": td.feasible,
        "Objective": round(td.objective, 2) if td.feasible else None,
        "Energy (J)": round(td.total_energy, 1) if td.feasible else None,
        "Build time (s)": round(build_time, 3),
    })

df_init = pd.DataFrame(init_rows)
df_init

**Expected output: Table 1 — Initial tour comparison**

| Heuristic | Tour length | Feasible | Objective | Energy (J) | Build time (s) |
|-----------|-------------|----------|-----------|------------|----------------|
| R1 (nearest) | 25 | — | — | — | — |
| R2 (earliest) | 25 | — | — | — | — |
| R3 (rand+NN) | 25 | — | — | — | — |
| R4 (consumption) | varies | — | — | — | — |
| R5 (random) | 3 | — | — | — | — |
| R6 (info-eff) | 25 | — | — | — | — |

**Settings:** R101 (50), $\eta = 1$, random slopes, campaign-time calibration.

**Purpose:** Compare the quality and feasibility of initial solutions before local search.
- R1/R2/R3 are geometry-based — fast but information-unaware.
- R4 is resource-aware — adapts tour length to budget.
- R5 is short and random — provides diversity for multi-start.
- R6 is information-and-energy-aware — expected to produce the best starting objective.
- Infeasible initial tours indicate the heuristic overcommitted the budget.

### 1b. Initial Tour Comparison Across Datasets

Run all heuristics on all datasets to check robustness.

In [ ]:
cross_rows = []
for ds_name, path in datasets.items():
    instance, graph, uav = make_instance(path)
    for h_name, builder in INITIAL_TOURS.items():
        tour_graph = builder(instance)
        solver = Solver(tour_graph, instance)
        td = solver.get_tour_data()
        cross_rows.append({
            "Instance": ds_name,
            "Heuristic": h_name,
            "Tour length": len(tour_graph.nodes) - 1,
            "Feasible": td.feasible,
            "Objective": round(td.objective, 2) if td.feasible else None,
        })

df_cross = pd.DataFrame(cross_rows)
# Pivot for readability
df_pivot = df_cross.pivot_table(index='Instance', columns='Heuristic',
                                 values='Objective', aggfunc='first')
df_pivot

**Expected output: Table 2 — Initial tour objectives across datasets**

| Instance | R1 | R2 | R3 | R4 | R5 | R6 |
|----------|----|----|----|----|----|----|  
| C101 (50) | — | — | — | — | — | — |
| R101 (50) | — | — | — | — | — | — |
| RC101 (100) | — | — | — | — | — | — |
| R101 (100) | — | — | — | — | — | — |
| C101 (100) | — | — | — | — | — | — |

**Purpose:** Verify that initial tour quality is consistent across instance types (clustered, random, mixed). R6 should consistently rank near the top. R4 may produce longer tours on easier instances. Infeasibility patterns reveal which heuristics are brittle on which instance geometries.

---
## 2. ILS: Effect of Initial Tour on Final Solution

Run ILS from each initial tour and compare final objectives.
This tests whether a better starting point leads to a better final solution,
or whether ILS is robust to initialization.

In [ ]:
ILS_STEPS = 200
ILS_T_IMPROVE = 30
ILS_K_REMOVE = 2
ILS_TABU_TENURE = 10

instance, graph, uav = make_instance(datasets["R101 (50)"])

ils_rows = []
for name, builder in INITIAL_TOURS.items():
    tour_graph = builder(instance)
    solver = Solver(tour_graph, instance)
    if solver.solution is None:
        ils_rows.append({"Initial tour": name, "Init obj": None, "Final obj": None,
                         "Improvement (%)": None, "Time (s)": None, "Final tour len": None})
        continue

    init_obj = solver.get_tour_data().objective
    state = State.initial_state(instance, tour_graph)
    optimizer = Optimizer(
        proposal=lambda s: universal_proposal(s),
        initial_state=state,
        maximize=True
    )
    t0 = time.time()
    for s in optimizer.run_ils(
        total_steps=ILS_STEPS,
        t_improve=ILS_T_IMPROVE,
        k_remove=ILS_K_REMOVE,
        tabu_tenure=ILS_TABU_TENURE,
    ):
        pass
    elapsed = time.time() - t0

    final_obj = optimizer._best_score
    improvement = ((final_obj - init_obj) / abs(init_obj) * 100) if init_obj else None
    ils_rows.append({
        "Initial tour": name,
        "Init obj": round(init_obj, 2),
        "Final obj": round(final_obj, 2) if final_obj else None,
        "Improvement (%)": round(improvement, 1) if improvement else None,
        "Time (s)": round(elapsed, 1),
        "Final tour len": len(list(optimizer._best_state.tour.nodes)) - 1 if optimizer._best_state else None,
    })

df_ils_init = pd.DataFrame(ils_rows)
df_ils_init

**Expected output: Table 3 — ILS from different initial tours**

| Initial tour | Init obj | Final obj | Improvement (%) | Time (s) | Final tour len |
|--------------|----------|-----------|-----------------|----------|----------------|
| R1 (nearest) | — | — | — | — | — |
| R2 (earliest) | — | — | — | — | — |
| R3 (rand+NN) | — | — | — | — | — |
| R4 (consumption) | — | — | — | — | — |
| R5 (random) | — | — | — | — | — |
| R6 (info-eff) | — | — | — | — | — |

**Settings:** R101 (50), $\eta = 1$, random slopes, ILS with 200 steps, $t_\text{improve} = 30$, $k_\text{remove} = 2$, tabu tenure = 10.

**Purpose:** Determine whether initial tour quality matters for ILS.
- If all final objectives converge to similar values → ILS is robust, initial tour mainly affects speed.
- If R6 consistently leads to better finals → initial tour quality propagates through search.
- R5 (short random) tests whether ILS can build up a good tour from a minimal start.
- Improvement (%) quantifies how much local search adds beyond the construction heuristic.

---
## 3. Metaheuristic Comparison

Compare ILS, Simulated Annealing, Hill Climbing, and Tilted Runs
on the same instance and initial tour, with the same step budget.

In [ ]:
NUM_STEPS = 300

instance, graph, uav = make_instance(datasets["R101 (50)"])
init_tour = build_initial_tour_R1(instance)  # same start for all

methods = {}

# --- ILS ---
state = State.initial_state(instance, init_tour)
opt = Optimizer(proposal=lambda s: universal_proposal(s), initial_state=state, maximize=True)
t0 = time.time()
history_ils = []
for s in opt.run_ils(total_steps=NUM_STEPS, t_improve=30, k_remove=2, tabu_tenure=10):
    history_ils.append(opt._best_score)
methods["ILS"] = {"obj": opt._best_score, "time": time.time() - t0, "history": history_ils}

# --- Simulated Annealing ---
state = State.initial_state(instance, init_tour)
opt = Optimizer(proposal=lambda s: universal_proposal(s), initial_state=state, maximize=True)
beta_fn = Optimizer.jumpcycle_beta_function(NUM_STEPS // 2, NUM_STEPS // 2)
t0 = time.time()
history_sa = []
for s in opt.simulated_annealing(num_steps=NUM_STEPS, beta_function=beta_fn, beta_magnitude=1.0):
    history_sa.append(opt._best_score)
methods["SA (jump)"] = {"obj": opt._best_score, "time": time.time() - t0, "history": history_sa}

# --- Hill Climbing ---
state = State.initial_state(instance, init_tour)
opt = Optimizer(proposal=lambda s: universal_proposal(s), initial_state=state, maximize=True)
t0 = time.time()
history_hc = []
for s in opt.ascent_run(num_steps=NUM_STEPS, accept=always_accept):
    history_hc.append(opt._best_score)
methods["Hill Climb"] = {"obj": opt._best_score, "time": time.time() - t0, "history": history_hc}

# --- Tilted Run (p=0.1) ---
state = State.initial_state(instance, init_tour)
opt = Optimizer(proposal=lambda s: universal_proposal(s), initial_state=state, maximize=True)
t0 = time.time()
history_tilt = []
for s in opt.tilted_run(num_steps=NUM_STEPS, p=0.1):
    history_tilt.append(opt._best_score)
methods["Tilted (p=0.1)"] = {"obj": opt._best_score, "time": time.time() - t0, "history": history_tilt}

meta_rows = [{"Method": k, "Best obj": round(v["obj"], 2) if v["obj"] else None,
              "Time (s)": round(v["time"], 1)} for k, v in methods.items()]
df_meta = pd.DataFrame(meta_rows)
df_meta

**Expected output: Table 4 — Metaheuristic comparison**

| Method | Best obj | Time (s) |
|--------|----------|----------|
| ILS | — | — |
| SA (jump) | — | — |
| Hill Climb | — | — |
| Tilted (p=0.1) | — | — |

**Settings:** R101 (50), $\eta = 1$, random slopes, 300 steps, R1 initial tour.

**Purpose:** Identify which metaheuristic is best suited for this problem.
- **Hill Climbing** is the baseline — fast but trapped in local optima.
- **Tilted run** accepts worse solutions with probability $p$ — mild diversification.
- **SA** uses temperature-based acceptance — more systematic exploration.
- **ILS** combines local search with perturbation + tabu — expected to be most effective because the perturbation mechanism removes multiple nodes and the tabu list prevents cycling.
- If ILS and SA converge to similar objectives, the simpler method is preferred.

### 3b. Convergence Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, data in methods.items():
    ax.plot(data["history"], label=name)
ax.set_xlabel("Iteration")
ax.set_ylabel("Best objective")
ax.set_title("Convergence: Metaheuristic Comparison — R101 (50)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig

**Expected output: Figure 1 — Convergence curves**

Four curves (ILS, SA, Hill Climb, Tilted) showing best objective vs iteration.

**Purpose:** Visualize convergence speed and final quality. Hill climbing should plateau earliest. SA and tilted runs may show staircase patterns. ILS should show jumps after perturbation phases.

---
## 4. ILS Hyperparameter Tuning

Sweep key ILS parameters: perturbation size ($k_\text{remove}$), stagnation threshold ($t_\text{improve}$), and tabu tenure.

In [ ]:
ILS_TOTAL = 300
instance, graph, uav = make_instance(datasets["R101 (50)"])
init_tour = build_initial_tour_R1(instance)

# Sweep k_remove
hp_rows = []
for k_rem in [1, 2, 3, 4]:
    for t_imp in [15, 30, 50]:
        for tenure in [5, 10, 20]:
            state = State.initial_state(instance, init_tour)
            opt = Optimizer(proposal=lambda s: universal_proposal(s),
                            initial_state=state, maximize=True)
            t0 = time.time()
            for s in opt.run_ils(total_steps=ILS_TOTAL, t_improve=t_imp,
                                 k_remove=k_rem, tabu_tenure=tenure):
                pass
            elapsed = time.time() - t0
            hp_rows.append({
                "k_remove": k_rem,
                "t_improve": t_imp,
                "tabu_tenure": tenure,
                "Best obj": round(opt._best_score, 2) if opt._best_score else None,
                "Time (s)": round(elapsed, 1),
            })

df_hp = pd.DataFrame(hp_rows)
df_hp.sort_values("Best obj", ascending=False).head(10)

**Expected output: Table 5 — ILS hyperparameter grid (top 10)**

| $k_\text{remove}$ | $t_\text{improve}$ | tabu tenure | Best obj | Time (s) |
|----------|-------------|-------------|----------|----------|
| — | — | — | — | — |
| ... | ... | ... | ... | ... |

**Settings:** R101 (50), $\eta = 1$, random slopes, 300 ILS steps, R1 initial tour.

**Grid:** $k_\text{remove} \in \{1, 2, 3, 4\}$, $t_\text{improve} \in \{15, 30, 50\}$, tabu tenure $\in \{5, 10, 20\}$ (36 combinations).

**Purpose:** Identify the best hyperparameter combination for ILS.
- **$k_\text{remove}$:** Too small → weak perturbation, stuck in basin. Too large → destroys good structure.
- **$t_\text{improve}$:** Too small → perturbations too frequent, search is erratic. Too large → wastes steps in stagnation.
- **Tabu tenure:** Too short → removed nodes re-enter immediately, perturbation is nullified. Too long → search space artificially restricted.
- Analogous to Tural et al.'s ILS hyperparameter tuning (Table 3 in their paper).

---
## 5. SA Temperature Schedule Comparison

Compare different $\beta$-function shapes for simulated annealing.

In [ ]:
SA_STEPS = 300
instance, graph, uav = make_instance(datasets["R101 (50)"])
init_tour = build_initial_tour_R1(instance)

schedules = {
    "jump-cycle": Optimizer.jumpcycle_beta_function(SA_STEPS // 2, SA_STEPS // 2),
    "linear-cycle": Optimizer.linearcycle_beta_function(SA_STEPS // 4, SA_STEPS // 4, SA_STEPS // 2),
    "logit-cycle": Optimizer.logitcycle_beta_function(SA_STEPS // 4, SA_STEPS // 4, SA_STEPS // 2),
}

sa_results = {}
for sched_name, beta_fn in schedules.items():
    state = State.initial_state(instance, init_tour)
    opt = Optimizer(proposal=lambda s: universal_proposal(s),
                    initial_state=state, maximize=True)
    t0 = time.time()
    history = []
    for s in opt.simulated_annealing(num_steps=SA_STEPS, beta_function=beta_fn, beta_magnitude=1.0):
        history.append(opt._best_score)
    elapsed = time.time() - t0
    sa_results[sched_name] = {"obj": opt._best_score, "time": elapsed, "history": history}

sa_rows = [{"Schedule": k, "Best obj": round(v["obj"], 2) if v["obj"] else None,
            "Time (s)": round(v["time"], 1)} for k, v in sa_results.items()]
pd.DataFrame(sa_rows)

**Expected output: Table 6 — SA schedule comparison**

| Schedule | Best obj | Time (s) |
|----------|----------|----------|
| jump-cycle | — | — |
| linear-cycle | — | — |
| logit-cycle | — | — |

**Settings:** R101 (50), $\eta = 1$, random slopes, 300 SA steps, R1 initial tour.

**Purpose:** Determine which cooling schedule works best.
- **Jump-cycle:** Alternates hot/cold phases abruptly — good for escaping local optima in bursts.
- **Linear-cycle:** Smooth cooldown then cold phase — more gradual.
- **Logit-cycle:** S-shaped transition — spends more time at intermediate temperatures.

---
## 6. Scalability: ILS vs Exact

Compare ILS solution quality against exact solver (from exact.ipynb) on all datasets.
ILS runs with fixed step budget; exact runs with 600s time limit.

In [ ]:
from uav_routing.solver.exact import solve_model_gurobi
import gurobipy as gp

ILS_STEPS_SCALE = 500
EXACT_TIME_LIMIT = 600

scale_rows = []
with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()

    for ds_name, path in datasets.items():
        instance, graph, uav = make_instance(path)
        n = len(graph.nodes)

        # --- Exact ---
        res_exact = solve_model_gurobi(instance, seed=SEED, time_limit=EXACT_TIME_LIMIT,
                                       env=env, prune=False, stats=False)
        exact_obj = res_exact["obj"] if res_exact and res_exact.get("obj") else None
        exact_time = res_exact["solve_time"] if res_exact else None
        exact_gap = res_exact.get("gap") if res_exact else None

        # --- ILS ---
        init_tour = build_initial_tour_R1(instance)
        state = State.initial_state(instance, init_tour)
        opt = Optimizer(proposal=lambda s: universal_proposal(s),
                        initial_state=state, maximize=True)
        t0 = time.time()
        for s in opt.run_ils(total_steps=ILS_STEPS_SCALE, t_improve=30,
                             k_remove=2, tabu_tenure=10):
            pass
        ils_time = time.time() - t0
        ils_obj = opt._best_score

        gap_to_exact = None
        if exact_obj and ils_obj:
            gap_to_exact = (exact_obj - ils_obj) / exact_obj * 100

        scale_rows.append({
            "Instance": ds_name,
            "n": n,
            "Exact obj": round(exact_obj, 2) if exact_obj else None,
            "Exact gap (%)": f"{exact_gap*100:.2f}" if exact_gap is not None else "-",
            "Exact time (s)": round(exact_time, 1) if exact_time else "-",
            "ILS obj": round(ils_obj, 2) if ils_obj else None,
            "ILS time (s)": round(ils_time, 1),
            "ILS gap to exact (%)": round(gap_to_exact, 2) if gap_to_exact is not None else "-",
        })

df_scale = pd.DataFrame(scale_rows)
df_scale

**Expected output: Table 7 — ILS vs Exact solver**

| Instance | $n$ | Exact obj | Exact gap (%) | Exact time (s) | ILS obj | ILS time (s) | ILS gap to exact (%) |
|----------|-----|-----------|---------------|-----------------|---------|--------------|----------------------|
| C101 (50) | 51 | — | — | — | — | — | — |
| R101 (50) | 51 | — | — | — | — | — | — |
| RC101 (100) | 101 | — | — | — | — | — | — |
| R101 (100) | 101 | — | — | — | — | — | — |
| C101 (100) | 101 | — | — | — | — | — | — |

**Settings:** $\eta = 1$, random slopes, campaign-time calibration. Exact: 600s time limit. ILS: 500 steps, R1 initial tour.

**Purpose:** The central comparison table.
- On 50-node instances: exact solver should find optimal; ILS gap quantifies heuristic quality.
- On 100-node instances: exact solver hits time limit with large gap; ILS may actually find better incumbents faster.
- ILS time should be much less than exact time on large instances.
- This is analogous to Tural et al.'s Table 5 (GI vs ILS vs Exact comparison).

---
## 7. ILS Under Different Slope Regimes

Does the slope regime affect local search as much as it affects the exact solver?

In [ ]:
ILS_STEPS_SLOPE = 300
test_path = datasets["R101 (50)"]

slope_rows = []
for slope in ['random', 'positive', 'zero']:
    instance, graph, uav = make_instance(test_path, slope=slope)
    init_tour = build_initial_tour_R1(instance)
    state = State.initial_state(instance, init_tour)
    opt = Optimizer(proposal=lambda s: universal_proposal(s),
                    initial_state=state, maximize=True)
    t0 = time.time()
    for s in opt.run_ils(total_steps=ILS_STEPS_SLOPE, t_improve=30,
                         k_remove=2, tabu_tenure=10):
        pass
    elapsed = time.time() - t0
    slope_rows.append({
        "Slope": slope,
        "Init obj": round(State.initial_state(instance, init_tour).value, 2),
        "Final obj": round(opt._best_score, 2) if opt._best_score else None,
        "Time (s)": round(elapsed, 1),
        "Final tour len": len(list(opt._best_state.tour.nodes)) - 1 if opt._best_state else None,
    })

df_ils_slopes = pd.DataFrame(slope_rows)
df_ils_slopes

**Expected output: Table 8 — ILS under different slope regimes**

| Slope | Init obj | Final obj | Time (s) | Final tour len |
|-------|----------|-----------|----------|----------------|
| random | — | — | — | — |
| positive | — | — | — | — |
| zero | — | — | — | — |

**Settings:** R101 (50), $\eta = 1$, ILS 300 steps, R1 initial tour.

**Purpose:** Compare with exact solver slope results (exact.ipynb Table 2).
- **Zero slopes** should have highest objective (all info is static, purely additive) but may show less improvement from search (timing irrelevant).
- **Positive slopes** should have highest improvement from search (ILS can optimize arrival timing to maximize growth).
- **Random slopes** test the general case where the search must balance early and late arrivals.

---
## 8. ILS Energy Sensitivity ($\eta$ sweep)

How does ILS perform under tighter energy budgets?

In [ ]:
ILS_STEPS_ETA = 300
test_path = datasets["R101 (50)"]

eta_rows = []
for eta in [0.3, 0.4, 0.6, 0.8, 1.0]:
    instance, graph, uav = make_instance(test_path, eta=eta)
    init_tour = build_initial_tour_R1(instance)
    solver_check = Solver(init_tour, instance)
    if solver_check.solution is None:
        # Initial tour infeasible at this eta — try R5 (shorter)
        init_tour = build_initial_tour_R5(instance)
        solver_check = Solver(init_tour, instance)

    if solver_check.solution is None:
        eta_rows.append({"eta": eta, "Init obj": None, "Final obj": None,
                         "Time (s)": None, "Final tour len": None})
        continue

    init_obj = solver_check.get_tour_data().objective
    state = State.initial_state(instance, init_tour)
    opt = Optimizer(proposal=lambda s: universal_proposal(s),
                    initial_state=state, maximize=True)
    t0 = time.time()
    for s in opt.run_ils(total_steps=ILS_STEPS_ETA, t_improve=30,
                         k_remove=2, tabu_tenure=10):
        pass
    elapsed = time.time() - t0
    eta_rows.append({
        "eta": eta,
        "Init obj": round(init_obj, 2),
        "Final obj": round(opt._best_score, 2) if opt._best_score else None,
        "Time (s)": round(elapsed, 1),
        "Final tour len": len(list(opt._best_state.tour.nodes)) - 1 if opt._best_state else None,
    })

df_ils_eta = pd.DataFrame(eta_rows)
df_ils_eta

**Expected output: Table 9 — ILS $\eta$ sensitivity**

| $\eta$ | Init obj | Final obj | Time (s) | Final tour len |
|--------|----------|-----------|----------|----------------|
| 0.3 | — | — | — | — |
| 0.4 | — | — | — | — |
| 0.6 | — | — | — | — |
| 0.8 | — | — | — | — |
| 1.0 | — | — | — | — |

**Settings:** R101 (50), random slopes, ILS 300 steps, R1 initial tour (falls back to R5 if infeasible).

**Purpose:** Compare with exact solver $\eta$ sweep (exact.ipynb Table 3).
- At tight $\eta$ (0.3–0.4), the initial tour may be infeasible at length $n/2$ — the fallback to R5 tests robustness.
- Tour length should decrease as $\eta$ decreases (fewer nodes affordable).
- ILS improvement over init should be larger at moderate $\eta$ (enough energy for some flexibility but not so much that the initial tour is already near-optimal).

---
## 9. Multi-Start ILS

Run ILS from all 6 initial tours and keep the global best.
This is the recommended production configuration.

In [ ]:
ILS_STEPS_MS = 200  # per start

ms_rows = []
for ds_name, path in datasets.items():
    instance, graph, uav = make_instance(path)
    best_obj_global = -float('inf')
    best_tour_name = None
    total_time = 0

    per_start = []
    for h_name, builder in INITIAL_TOURS.items():
        tour_graph = builder(instance)
        solver_check = Solver(tour_graph, instance)
        if solver_check.solution is None:
            per_start.append({"start": h_name, "obj": None})
            continue

        state = State.initial_state(instance, tour_graph)
        opt = Optimizer(proposal=lambda s: universal_proposal(s),
                        initial_state=state, maximize=True)
        t0 = time.time()
        for s in opt.run_ils(total_steps=ILS_STEPS_MS, t_improve=30,
                             k_remove=2, tabu_tenure=10):
            pass
        elapsed = time.time() - t0
        total_time += elapsed
        per_start.append({"start": h_name, "obj": opt._best_score})

        if opt._best_score and opt._best_score > best_obj_global:
            best_obj_global = opt._best_score
            best_tour_name = h_name

    ms_rows.append({
        "Instance": ds_name,
        "n": len(graph.nodes),
        "Best obj": round(best_obj_global, 2) if best_obj_global > -float('inf') else None,
        "Best start": best_tour_name,
        "Total time (s)": round(total_time, 1),
    })

df_ms = pd.DataFrame(ms_rows)
df_ms

**Expected output: Table 10 — Multi-start ILS**

| Instance | $n$ | Best obj | Best start | Total time (s) |
|----------|-----|----------|------------|----------------|
| C101 (50) | 51 | — | — | — |
| R101 (50) | 51 | — | — | — |
| RC101 (100) | 101 | — | — | — |
| R101 (100) | 101 | — | — | — |
| C101 (100) | 101 | — | — | — |

**Settings:** $\eta = 1$, random slopes, 200 ILS steps per start, 6 starts (R1–R6).

**Purpose:** This is the full multi-start ILS as described in the paper.
- "Best start" shows which initial tour led to the global best — if R6 consistently wins, it validates the information-efficiency heuristic.
- Total time = sum of all 6 ILS runs. Compare with exact solver time from Table 7.
- On 100-node instances, multi-start ILS should be faster than exact and potentially find better incumbents.

---
## 10. Reproducibility

5 runs with different random seeds to check variance in ILS results.

In [ ]:
ILS_STEPS_REPRO = 300
instance, graph, uav = make_instance(datasets["R101 (50)"])

repro_rows = []
for seed_val in [42, 123, 456, 789, 1024]:
    rng = random.Random(seed_val)
    random.seed(seed_val)  # global seed for proposal operators
    init_tour = build_initial_tour_R3(instance, rng=rng)  # R3 uses randomness
    state = State.initial_state(instance, init_tour)
    opt = Optimizer(proposal=lambda s: universal_proposal(s),
                    initial_state=state, maximize=True)
    t0 = time.time()
    for s in opt.run_ils(total_steps=ILS_STEPS_REPRO, t_improve=30,
                         k_remove=2, tabu_tenure=10):
        pass
    elapsed = time.time() - t0
    repro_rows.append({
        "Seed": seed_val,
        "Best obj": round(opt._best_score, 2) if opt._best_score else None,
        "Time (s)": round(elapsed, 1),
        "Final tour len": len(list(opt._best_state.tour.nodes)) - 1 if opt._best_state else None,
    })

df_repro = pd.DataFrame(repro_rows)
print(df_repro.to_string(index=False))
objs = df_repro['Best obj'].dropna()
print(f"\nObjective: mean={objs.mean():.2f}, std={objs.std():.2f}")

**Expected output: Table 11 — ILS reproducibility**

| Seed | Best obj | Time (s) | Final tour len |
|------|----------|----------|----------------|
| 42 | — | — | — |
| 123 | — | — | — |
| 456 | — | — | — |
| 789 | — | — | — |
| 1024 | — | — | — |
| **mean ± std** | — | — | — |

**Settings:** R101 (50), $\eta = 1$, random slopes, ILS 300 steps, R3 initial tour (random first node).

**Purpose:** Quantify stochastic variance in ILS.
- Small std → ILS is reliable, single runs are representative.
- Large std → multi-start is essential, and results should be reported as mean ± std.
- Uses R3 (random-then-NN) to introduce variance in both initial tour and search trajectory.
- Compare with exact solver reproducibility (exact.ipynb Table 5) — ILS should have higher variance since it is stochastic.